In [ ]:
%%capture
import os
import modin.pandas as pd
from dj_notebook import activate
from pathlib import Path
env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
from edc_pdutils.dataframes import get_subject_visit
from edc_appointment.analytics import get_appointment_df
from intecomm_analytics.dataframes import get_df_main_1858, get_appt_df


In [ ]:
df_main_1858 = get_df_main_1858(None, fasting_hours=8.0)

In [ ]:
df_main_1858

In [ ]:
df_visit = get_subject_visit("intecomm_subject.subjectvisit")

In [ ]:
df_appointment = get_appointment_df()

In [ ]:
df_appointment.query("appt_status.isin(['done', 'incomplete'])").appt_status.value_counts()

In [ ]:
df_appointment.query("appt_status.isin(['skipped', 'cancelled'])").appt_status.value_counts()


In [ ]:
# df_appointment.query("appt_status.isin(['done', 'incomplete']) and appt_timing=='missed'").groupby("").appt_status.value_counts()

In [ ]:
from intecomm_analytics.dataframes.df_main_1858.get_df_main_1858_pre import merge_in_rando
# from intecomm_analytics.dataframes import get_patientlog_df, get_df_main_1858
df_appt = get_appt_df()
# df_main = get_patientlog_df()
# # exclude those in patient_log that were not added to a group
# df_main = df_main[(df_main.group_identifier.notna())]
#
# # exclude those added to a group but never consented
# df_main = df_main[(df_main.consent_datetime.notna())]
#
# assert len(df_main) == 1864  # nosec B101
#
# # rename conditions reported at screening to distinguish from those
# # confirmed later at baseline
# df_main.rename(columns={"hiv": "hiv_scr", "htn": "htn_scr", "dm": "dm_scr"}, inplace=True)
#
# df_main = merge_in_rando(df_main)
# df_appointment = df_appointment.merge(get_df_main_1858(None, fasting_hours=8.0), how="left", on="subject_identifier")
# df_appointment

In [ ]:
df = df_appt.query("appt_status.isin(['done', 'incomplete']) and appt_timing=='missed' and primary_cohort_str.isin(['HTN_ALONE', 'HTN_DM', 'DM_ALONE'])").assignment.value_counts().to_frame().reset_index()

In [ ]:
# df = df.drop(columns=[""])
df = df.set_index('assignment').T.reset_index(drop=True)
df.index.name = ''
df['label'] = "Missed appointments"
df.reset_index(drop=True)

In [ ]:
df.pivot(columns='assignment', values='count')

In [ ]:
df_appt.query("appt_status.isin(['done', 'incomplete']) and appt_timing=='missed' and primary_cohort_str.isin(['HTN_ALONE', 'HTN_DM', 'DM_ALONE'])").groupby("assignment").size()

In [ ]:
# Number of participants who missed one or more appointments; community
len(df_appt.query("appt_status.isin(['done', 'incomplete']) and appt_timing=='missed' and primary_cohort_str.isin(['HTN_ALONE', 'HTN_DM', 'DM_ALONE']) and assignment=='a'").subject_identifier.unique())

In [ ]:
# Number of participants who missed one or more appointments; facility
len(df_appt.query("appt_status.isin(['done', 'incomplete']) and appt_timing=='missed' and primary_cohort_str.isin(['HTN_ALONE', 'HTN_DM', 'DM_ALONE']) and assignment=='b'").subject_identifier.unique())


In [ ]:
df_appt.query("appt_status.isin(['done', 'incomplete']) and appt_timing=='missed' and primary_cohort_str.isin(['HIV_ALONE'])").groupby("assignment").size()


In [ ]:
# Number of participants who missed one or more appointments; community
len(df_appt.query("appt_status.isin(['done', 'incomplete']) and appt_timing=='missed' and primary_cohort_str.isin(['HIV_ALONE']) and assignment=='a'").subject_identifier.unique())

In [ ]:
# Number of participants who missed one or more appointments; facility
value = len(df_appt.query("appt_status.isin(['done', 'incomplete']) and appt_timing=='missed' and primary_cohort_str.isin(['HIV_ALONE']) and assignment=='b'").subject_identifier.unique())
df = pd.DataFrame(data={"label": ["Number of participants who missed one or more appointments"], "assignment": ["b"], "value": value})
df

df_out= pd.concat([df_out, df])

In [ ]:
# df_log = get_patientlog_df()

In [ ]:
df_log
